# Batch normalization

_Deep Learning_

**Rescale each layer's inputs to be tidy and centered, so training is faster and smoother.**

As data flows through layers, its scale can drift, getting too big or too small. That slows training.

     Batch normalization cleans up a layer's inputs: it centers them at 0 and scales them to a steady size.

---

This notebook builds the idea up one step at a time: first watch a BatchNorm layer flatten messy inputs to mean 0 / std 1, then see on real tumor data why feeding a network well-scaled inputs makes training steadier. Run each cell and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PyTorch

We'll build a tiny network with a `BatchNorm1d` layer, push deliberately messy inputs through it, and confirm that BatchNorm recenters and rescales those inputs to roughly mean 0 and std 1.

### Step 1 — Build a network with a BatchNorm layer

The network is a `Linear → BatchNorm1d → ReLU → Linear` stack. The `BatchNorm1d(32)` layer sits right after the first linear layer and normalizes its 32 outputs **across the batch** — for each feature it subtracts the batch mean and divides by the batch standard deviation.

In [ ]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(16, 32),
    nn.BatchNorm1d(32),       # normalize the 32 features across the batch
    nn.ReLU(),
    nn.Linear(32, 1),
)

### Step 2 — Feed in deliberately messy inputs

We create a batch of 64 random inputs and shift them so they have a large mean (~10) and a big spread (std ~5). This mimics the kind of badly-scaled activations that drift through a deep network — exactly the situation BatchNorm is meant to fix.

In [ ]:
x = torch.randn(64, 16) * 5 + 10   # messy inputs: large mean (~10), big spread (std ~5)

### Step 3 — Check the output is centered and rescaled

We put the network in `train()` mode (so BatchNorm uses the current batch's statistics), then run the input through just the first linear layer and the BatchNorm layer. Despite the messy input, the BatchNorm output comes out at roughly mean 0 and std 1 — tidy, centered, and steady.

In [ ]:
net.train()                       # train mode: BatchNorm uses this batch's statistics

linear_out = net[0](x)            # output of the first Linear layer
bn_out = net[1](linear_out)       # output of the BatchNorm layer

print("after BN mean:", round(bn_out.mean().item(), 3))   # ~ 0
print("after BN std: ", round(bn_out.std().item(), 3))    # ~ 1

## Visualize the data & results

_On real tumor data, does normalizing the inputs (batch-norm's job) let the network train steadily?_

### Step 1 — Prepare raw and normalized versions of the data

We load the breast-cancer dataset twice: once with the **raw** features and once **standardized** to mean 0 / std 1. Both go through the same train/test split (same `random_state`, so the rows line up), giving us a clean before/after comparison of what normalization does.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss

bc = load_breast_cancer()
Xn = StandardScaler().fit_transform(bc.data)      # normalized copy: mean 0, std 1

# Raw (unscaled) split.
Xtr_r, Xte_r, ytr, yte = train_test_split(bc.data, bc.target, test_size=0.3,
                                          random_state=0, stratify=bc.target)
# Normalized split, same random_state so rows line up.
Xtr_s, Xte_s, _, _ = train_test_split(Xn, bc.target, test_size=0.3,
                                      random_state=0, stratify=bc.target)

### Step 2 — Train a network and record its validation loss each epoch

This helper trains the same small network for 40 epochs, one epoch at a time (`partial_fit`), recording the validation log loss after each. Calling it on the raw inputs and on the normalized inputs lets us plot two learning curves on the same axes.

In [ ]:
def val_curve(Xt, Xv):
    m = MLPClassifier(hidden_layer_sizes=(64, 64), solver="adam", max_iter=1,
                      warm_start=True, random_state=0, alpha=1e-4,
                      learning_rate_init=0.003)
    va = []
    for e in range(40):
        if e == 0:
            m.partial_fit(Xt, ytr, classes=[0, 1])   # first call declares the classes
        else:
            m.partial_fit(Xt, ytr)                   # subsequent epochs
        va.append(log_loss(yte, m.predict_proba(Xv), labels=[0, 1]))
    return va

raw_curve = val_curve(Xtr_r, Xte_r)
norm_curve = val_curve(Xtr_s, Xte_s)

### Step 3 — Plot raw vs normalized learning curves

We plot both validation-loss curves together. The normalized inputs train faster and settle to a lower, smoother loss, while the raw inputs lag and wobble — the same steadying effect BatchNorm provides *inside* a network, shown here at its input.

In [ ]:
plt.plot(raw_curve, color="#ff7b72", label="raw inputs (unscaled)")
plt.plot(norm_curve, color="#7ee787", label="normalized inputs")
plt.title("Real validation loss: raw vs normalized breast-cancer inputs")
plt.xlabel("epoch")
plt.ylabel("log loss")
plt.legend()
plt.show()